# 04 – Graph (Neo4j) – Data preparation
Målet är att förbereda en grafmodell från våra e-commerce data.

Vi använder **Databricks/Spark** för att läsa och transformera data och skapar:

- **Noder:** `Customer`, `Product`
- **Relation:** `(Customer)-[:BOUGHT]->(Product)`

Resultatet exporteras som **CSV** och importeras i **Neo4j Desktop**, där vi kör **Cypher** för grafanalys.

## Grafmodell
- `Customer { customer_id }`
- `Product { product_id }`
- `(Customer)-[:BOUGHT { times_bought }]->(Product)`

## 1. Läs in raw-data

In [0]:
from pyspark.sql import functions as F

base_path = "dbfs:/Workspace/Repos/sumeyacabuk@hotmail.com/beauty_lakehouse/data/raw/"

customers_df = spark.read.option("header", True).csv(base_path + "customers.csv")
products_df  = spark.read.option("header", True).csv(base_path + "products.csv")
orders_df    = spark.read.option("header", True).csv(base_path + "orders.csv")
order_items_df = spark.read.option("header", True).csv(base_path + "order_items.csv")

display(customers_df.limit(5))
display(products_df.limit(5))
display(orders_df.limit(5))
display(order_items_df.limit(5))

## 2. Kontrollera schema

In [0]:
customers_df.printSchema()
products_df.printSchema()
orders_df.printSchema()
order_items_df.printSchema()

## 3. Bygg relationen `BOUGHT`
Vi bygger edges **Customer → Product** baserat på `order_items` + `orders` och räknar hur många gånger en kund köpt en produkt (`times_bought`).


In [0]:
# Bygger edges: Customer -> Product baserat på order_items + orders
bought_edges = (
    order_items_df
    .join(orders_df.select("order_id", "customer_id"), on="order_id", how="inner")
    .select(
        F.col("customer_id").cast("string").alias("customer_id"),
        F.col("product_id").cast("string").alias("product_id")
    )
    .groupBy("customer_id", "product_id")
    .agg(F.count("*").alias("times_bought"))
)

display(bought_edges.orderBy(F.desc("times_bought")).limit(10))

## 4. Snabb analys
Topplista: produkter som köpts mest (summa `times_bought` per `product_id`).

In [0]:
display(
    bought_edges
    .groupBy("product_id")
    .sum("times_bought")
    .orderBy("sum(times_bought)", ascending=False)
    .limit(10)
)

## 5. Skapa node-listor
Vi tar fram unika noder för `Customer` och `Product` som Neo4j kan importera.

In [0]:
customers_nodes = customers_df.select(F.col("customer_id").cast("string").alias("customer_id")).dropDuplicates()
products_nodes  = products_df.select(F.col("product_id").cast("string").alias("product_id")).dropDuplicates()

print("Customers:", customers_nodes.count())
print("Products:", products_nodes.count())
print("Edges:", bought_edges.count())

## 6. Export till CSV för Neo4j
Vi exporterar noder och relationer som CSV till Unity Catalog Volume.

> Obs: `coalesce(1)` ger en (1) fil per dataset (enklare att ladda ner/importera).

In [0]:
display(bought_edges.orderBy(F.desc("times_bought")).limit(10))

## 6. Export till CSV för Neo4j
Vi exporterar noder och relationer som CSV till Unity Catalog Volume.

> Obs: `coalesce(1)` ger en (1) fil per dataset (enklare att ladda ner/importera).

In [0]:
export_base = "/Volumes/workspace/beauty/data/neo4j_export"

(customers_nodes.coalesce(1)
 .write.mode("overwrite").option("header", True)
 .csv(export_base + "/customers_nodes"))

(products_nodes.coalesce(1)
 .write.mode("overwrite").option("header", True)
 .csv(export_base + "/products_nodes"))

(bought_edges.coalesce(1)
 .write.mode("overwrite").option("header", True)
 .csv(export_base + "/bought_edges"))

## 7. Visa filer att ladda ner
Här ser du vilka filer som skapats och kan laddas ner för import i Neo4j Desktop.

In [0]:
dbutils.fs.ls(export_base + "/customers_nodes")
dbutils.fs.ls(export_base + "/products_nodes")
dbutils.fs.ls(export_base + "/bought_edges")

In [0]:
bought_edges.show(10)